In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D1 — CEDEFOP Labour Skills Shortage Dataset
# ============================================================

import hashlib
import json
import platform
import re
import sys
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
from openpyxl.utils.cell import range_boundaries

In [2]:
# ------------------------------------------------------------
# 1. Source upload and configuration
# ------------------------------------------------------------

from google.colab import files

uploaded = files.upload()

SOURCE_PATH = Path(next(iter(uploaded)))

DOCUMENT_ID = "D1"
DOCUMENT_NAME = "CEDEFOP Labour Skills Shortage Dataset"

OUTPUT_DIR = Path(f"outputs_{DOCUMENT_ID}_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded source file: {SOURCE_PATH}")
print(f"Stage 1 output directory: {OUTPUT_DIR}")

Saving D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx to D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx
Loaded source file: D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx
Stage 1 output directory: outputs_D1_stage1


In [3]:
# ------------------------------------------------------------
# 2. Source identity and SHA-256 traceability
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

print(f"Source file: {SOURCE_PATH.name}")
print(f"SHA-256: {SOURCE_SHA256}")

Source file: D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx
SHA-256: 4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389


In [4]:
# ------------------------------------------------------------
# 3. Workbook loading and metadata
# ------------------------------------------------------------

wb = openpyxl.load_workbook(
    SOURCE_PATH,
    data_only=True
)

sheet_names = wb.sheetnames

print("Workbook loaded successfully.")
print("Sheets:", sheet_names)

document_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_filename": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "file_format": SOURCE_PATH.suffix.replace(".", "").upper(),
    "number_of_sheets": len(sheet_names),
    "sheet_names": sheet_names,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "openpyxl_version": openpyxl.__version__
}

document_metadata

with open(
    OUTPUT_DIR / "D1_document_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        document_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

Workbook loaded successfully.
Sheets: ['EU27', 'IT', 'NL', 'PT']


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [5]:
# ------------------------------------------------------------
# 4. Worksheet characterisation
# ------------------------------------------------------------

sheet_characterisation = []

numeric_component_pattern = re.compile(
    r"^\s*-?\d+(?:[.,]\d+)?(?:\s*-\s*-?\d+(?:[.,]\d+)?)+\s*$"
)

for ws in wb.worksheets:
    merged_ranges = list(ws.merged_cells.ranges)

    non_empty_cells = 0
    numeric_cells = 0
    numeric_like_text_cells = 0
    text_cells = 0
    empty_cells = 0

    for row in ws.iter_rows():
        for cell in row:
            value = cell.value

            if value is None:
                empty_cells += 1
                continue

            non_empty_cells += 1

            if isinstance(value, (int, float)) and not isinstance(value, bool):
                numeric_cells += 1

            elif isinstance(value, str):
                text_cells += 1

                if numeric_component_pattern.match(value):
                    numeric_like_text_cells += 1

    total_cells = ws.max_row * ws.max_column

    native_numerical_density = (
        numeric_cells / non_empty_cells
        if non_empty_cells
        else 0
    )

    quantitative_content_density = (
        (numeric_cells + numeric_like_text_cells) / non_empty_cells
        if non_empty_cells
        else 0
    )

    sheet_characterisation.append({
        "sheet_name": ws.title,
        "max_rows": ws.max_row,
        "max_columns": ws.max_column,
        "total_cells": total_cells,
        "non_empty_cells": non_empty_cells,
        "empty_cells": empty_cells,
        "text_cells": text_cells,
        "numeric_cells": numeric_cells,
        "numeric_like_text_cells": numeric_like_text_cells,
        "native_numerical_density": round(native_numerical_density, 3),
        "quantitative_content_density": round(
            quantitative_content_density,
            3
        ),
        "merged_cell_count": len(merged_ranges),
        "merged_ranges": [str(rng) for rng in merged_ranges],
    })

sheet_characterisation_df = pd.DataFrame(sheet_characterisation)
sheet_characterisation_df

sheet_characterisation_df.to_csv(
    OUTPUT_DIR / "D1_sheet_characterisation.csv",
    index=False
)

In [6]:
# ------------------------------------------------------------
# 5. Merged-cell diagnostics
# ------------------------------------------------------------

merged_cell_diagnostics = []

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    merged_ranges = list(ws.merged_cells.ranges)

    for merged_range in merged_ranges:
        min_col, min_row, max_col, max_row = range_boundaries(str(merged_range))

        top_left_value = ws.cell(
            row=min_row,
            column=min_col
        ).value

        merged_cell_values = []

        for row in range(min_row, max_row + 1):
            for col in range(min_col, max_col + 1):
                merged_cell_values.append(
                    ws.cell(row=row, column=col).value
                )

        merged_cell_diagnostics.append({
            "Sheet": sheet_name,
            "Merged Range": str(merged_range),
            "Top-left Value": top_left_value,
            "Rows Spanned": max_row - min_row + 1,
            "Columns Spanned": max_col - min_col + 1,
            "Non-empty Cells in Range": sum(
                value is not None
                for value in merged_cell_values
            ),
            "Empty Cells in Range": sum(
                value is None
                for value in merged_cell_values
            )
        })

merged_cell_diagnostics_df = pd.DataFrame(
    merged_cell_diagnostics
)

merged_cell_diagnostics_df

merged_summary = (
    merged_cell_diagnostics_df
    .groupby("Sheet", as_index=False)
    .agg(
        Merged_Ranges=("Merged Range", "count"),
        Empty_Cells_Inside_Merged_Ranges=("Empty Cells in Range", "sum")
    )
)

merged_summary

merged_cell_diagnostics_df.to_csv(
    OUTPUT_DIR /
    "D1_merged_cell_diagnostics.csv",
    index=False
)

merged_summary.to_csv(
    OUTPUT_DIR /
    "D1_merged_cell_summary.csv",
    index=False
)

In [7]:
# ------------------------------------------------------------
# 6. Hierarchical merged-cell inspection
# ------------------------------------------------------------

hierarchical_merge_inspection = []

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    headers = {
        ws.cell(row=1, column=col).value: col
        for col in range(1, ws.max_column + 1)
    }

    main_group_col = headers.get("Main Occupation Group")

    if main_group_col is None:
        raise ValueError(
            f"'Main Occupation Group' column not found in sheet {sheet_name}."
        )

    relevant_ranges = []

    for merged_range in ws.merged_cells.ranges:
        min_col, _, max_col, _ = range_boundaries(
            str(merged_range)
        )

        if min_col <= main_group_col <= max_col:
            relevant_ranges.append(str(merged_range))

    hierarchical_merge_inspection.append({
        "sheet_name": sheet_name,
        "merged_range_count": len(relevant_ranges),
        "merged_ranges": "; ".join(relevant_ranges)
    })

hierarchical_merge_df = pd.DataFrame(
    hierarchical_merge_inspection
)

hierarchical_merge_df

hierarchical_merge_df.to_csv(
    OUTPUT_DIR / "D1_hierarchical_merge_inspection.csv",
    index=False
)

In [8]:
# ------------------------------------------------------------
# 7. Header structure inspection
# ------------------------------------------------------------

header_inspection = []

for ws in wb.worksheets:
    first_row = [
        ws.cell(row=1, column=col).value
        for col in range(1, ws.max_column + 1)
    ]

    non_null_headers = [h for h in first_row if h is not None]

    header_inspection.append({
        "sheet_name": ws.title,
        "assumed_header_row": 1,
        "headers": first_row,
        "header_count": len(first_row),
        "missing_headers": sum(h is None for h in first_row),
        "duplicate_headers": len(non_null_headers) - len(set(non_null_headers)),
    })

header_inspection_df = pd.DataFrame(header_inspection)
header_inspection_df

header_inspection_df.to_csv(
    OUTPUT_DIR / "D1_header_inspection.csv",
    index=False
)

In [9]:
# ------------------------------------------------------------
# 8. Structural blank analysis
# ------------------------------------------------------------

structural_blank_summary = []

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    headers = {
        ws.cell(row=1, column=col).value: col
        for col in range(1, ws.max_column + 1)
    }

    main_group_col = headers.get(
        "Main Occupation Group"
    )

    if main_group_col is None:
        raise ValueError(
            f"'Main Occupation Group' column not found "
            f"in sheet {sheet_name}."
        )

    relevant_merged_ranges = []
    structural_blank_count = 0

    for merged_range in ws.merged_cells.ranges:
        min_col, min_row, max_col, max_row = (
            range_boundaries(str(merged_range))
        )

        if min_col <= main_group_col <= max_col:
            relevant_merged_ranges.append(
                str(merged_range)
            )

            number_of_cells = (
                (max_row - min_row + 1)
                * (max_col - min_col + 1)
            )

            structural_blank_count += (
                number_of_cells - 1
            )

    structural_blank_summary.append({
        "sheet_name": sheet_name,
        "merged_ranges_in_main_occupation_group":
            "; ".join(relevant_merged_ranges),
        "main_occupation_group_structural_blanks":
            structural_blank_count,
        "interpretation":
            "Blank cells are generated by vertically merged "
            "hierarchical labels and are therefore treated as "
            "structural representation rather than missing "
            "source information."
    })

structural_blank_summary_df = pd.DataFrame(
    structural_blank_summary
)

structural_blank_summary_df

structural_blank_summary_df.to_csv(
    OUTPUT_DIR / "D1_structural_blank_summary.csv",
    index=False
)

In [10]:
# ------------------------------------------------------------
# 9. Reference dataset construction
# ------------------------------------------------------------

reference_rows = []

for sheet in sheet_names:
    df = pd.read_excel(
        SOURCE_PATH,
        sheet_name=sheet
    )

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    # Canonicalise the source header for reference construction only.
    df = df.rename(columns={
        "Labour Shortage Indexx":
            "Labour Shortage Index"
    })

    df = df.dropna(how="all")

    # Reconstruct hierarchical labels represented through merged cells.
    required_source_columns = [
        "Main Occupation Group",
        "Occupation Group (2 digit)",
        "Labour Shortage Index",
        "LSI (Comp.)",
        "LSI1",
        "LSI2",
        "LSI3"
    ]

    missing_source_columns = [
        col
        for col in required_source_columns
        if col not in df.columns
    ]

    if missing_source_columns:
        raise ValueError(
            f"Missing source columns in sheet {sheet}: "
            f"{missing_source_columns}"
        )

    df["Main Occupation Group"] = (
        df["Main Occupation Group"].ffill()
    )

    df.insert(
        0,
        "Geographic Area",
        sheet
    )

    reference_rows.append(df)

reference_values_df = pd.concat(
    reference_rows,
    ignore_index=True
)

for col in [
    "Main Occupation Group",
    "Occupation Group (2 digit)"
]:
    reference_values_df[col] = (
        reference_values_df[col]
        .astype(str)
        .str.strip()
        .replace("nan", np.nan)
    )

reference_values_df.head()

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Geographic Area,Main Occupation Group,Occupation Group (2 digit),Labour Shortage Index,LSI (Comp.),LSI1,LSI2,LSI3
0,EU27,High-skilled non-manual occupations,"Chief executives, senior officials and legisla...",3.000000,4-4-1,4,4,1
1,EU27,High-skilled non-manual occupations,Administrative and commercial managers,2.333333,4-2-1,4,2,1
2,EU27,High-skilled non-manual occupations,Production and specialised services managers,3.000000,3-4-2,3,4,2
3,EU27,High-skilled non-manual occupations,"Hospitality, retail and other services managers",2.666667,2-3-3,2,3,3
4,EU27,High-skilled non-manual occupations,Science and engineering professionals,2.666667,4-3-1,4,3,1


In [11]:
# ------------------------------------------------------------
# 10. Definition of extraction task
# ------------------------------------------------------------

EXTRACTION_TASK = """
For each worksheet in the workbook, extract the labour shortage information by occupation group.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

The output must preserve the association between each geographic area,
occupation group, and its corresponding LSI indicators.
"""

print(EXTRACTION_TASK)

with open(
    OUTPUT_DIR / f"{DOCUMENT_ID}_extraction_task.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(EXTRACTION_TASK.strip() + "\n")


For each worksheet in the workbook, extract the labour shortage information by occupation group.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

The output must preserve the association between each geographic area,
occupation group, and its corresponding LSI indicators.



In [12]:
# ------------------------------------------------------------
# 11. Extraction schema
# ------------------------------------------------------------

EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "records": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "Geographic Area": {
                        "type": ["string", "null"]
                    },
                    "Main Occupation Group": {
                        "type": ["string", "null"]
                    },
                    "Occupation Group (2 digit)": {
                        "type": ["string", "null"]
                    },
                    "Labour Shortage Index": {
                        "type": ["number", "null"]
                    },
                    "LSI (Comp.)": {
                        "type": ["string", "null"]
                    },
                    "LSI1": {
                        "type": ["number", "null"]
                    },
                    "LSI2": {
                        "type": ["number", "null"]
                    },
                    "LSI3": {
                        "type": ["number", "null"]
                    }
                },
                "required": [
                    "Geographic Area",
                    "Main Occupation Group",
                    "Occupation Group (2 digit)",
                    "Labour Shortage Index",
                    "LSI (Comp.)",
                    "LSI1",
                    "LSI2",
                    "LSI3"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["records"],
    "additionalProperties": False
}

with open(
    OUTPUT_DIR / "D1_extraction_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        EXTRACTION_SCHEMA,
        f,
        indent=2,
        ensure_ascii=False
    )

In [13]:
# ------------------------------------------------------------
# 12. Definition of reference-value schema
# ------------------------------------------------------------

REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": "labour_shortage_indicator_by_geographic_area_and_occupation_group",
    "fields": {
        "Geographic Area": "Worksheet or geographic-area identifier",
        "Main Occupation Group": "Broad occupation category",
        "Occupation Group (2 digit)": "Two-digit occupation group label",
        "Labour Shortage Index": "Composite labour shortage indicator",
        "LSI (Comp.)": "Component representation of the Labour Shortage Index",
        "LSI1": "First component indicator",
        "LSI2": "Second component indicator",
        "LSI3": "Third component indicator"
    }
}

print(json.dumps(REFERENCE_SCHEMA, indent=2))

{
  "document_id": "D1",
  "record_level": "labour_shortage_indicator_by_geographic_area_and_occupation_group",
  "fields": {
    "Geographic Area": "Worksheet or geographic-area identifier",
    "Main Occupation Group": "Broad occupation category",
    "Occupation Group (2 digit)": "Two-digit occupation group label",
    "Labour Shortage Index": "Composite labour shortage indicator",
    "LSI (Comp.)": "Component representation of the Labour Shortage Index",
    "LSI1": "First component indicator",
    "LSI2": "Second component indicator",
    "LSI3": "Third component indicator"
  }
}


In [14]:
# ------------------------------------------------------------
# 13. Validation of reference dataset structure
# ------------------------------------------------------------

expected_columns = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

missing_columns = [
    column
    for column in expected_columns
    if column not in reference_values_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}"
    )

reference_values_df = reference_values_df[
    expected_columns
]

reference_summary = {
    "number_of_reference_records":
        int(len(reference_values_df)),

    "geographic_areas": sorted(
        reference_values_df[
            "Geographic Area"
        ].dropna().unique().tolist()
    ),

    "number_of_geographic_areas":
        int(
            reference_values_df[
                "Geographic Area"
            ].nunique()
        ),

    "number_of_main_occupation_groups":
        int(
            reference_values_df[
                "Main Occupation Group"
            ].nunique()
        ),

    "number_of_occupation_groups":
        int(
            reference_values_df[
                "Occupation Group (2 digit)"
            ].nunique()
        ),

    "missing_values_by_column": {
        column: int(value)
        for column, value
        in reference_values_df
        .isna()
        .sum()
        .to_dict()
        .items()
    }
}

reference_summary

{'number_of_reference_records': 156,
 'geographic_areas': ['EU27', 'IT', 'NL', 'PT'],
 'number_of_geographic_areas': 4,
 'number_of_main_occupation_groups': 4,
 'number_of_occupation_groups': 40,
 'missing_values_by_column': {'Geographic Area': 0,
  'Main Occupation Group': 0,
  'Occupation Group (2 digit)': 0,
  'Labour Shortage Index': 0,
  'LSI (Comp.)': 0,
  'LSI1': 0,
  'LSI2': 0,
  'LSI3': 0}}

In [15]:
# ------------------------------------------------------------
# 14. Reference numerical consistency check
# ------------------------------------------------------------

numeric_lsi_columns = [
    "Labour Shortage Index",
    "LSI1",
    "LSI2",
    "LSI3"
]

for col in numeric_lsi_columns:
    reference_values_df[col] = pd.to_numeric(
        reference_values_df[col],
        errors="coerce"
    )

calculated_lsi = reference_values_df[
    ["LSI1", "LSI2", "LSI3"]
].mean(axis=1)

lsi_consistency_mask = np.isclose(
    reference_values_df["Labour Shortage Index"],
    calculated_lsi,
    atol=1e-6,
    equal_nan=False
)

lsi_consistency_summary = {
    "formula_checked": "Labour Shortage Index = mean(LSI1, LSI2, LSI3)",
    "records_checked": int(len(reference_values_df)),
    "consistent_records": int(lsi_consistency_mask.sum()),
    "inconsistent_records": int((~lsi_consistency_mask).sum()),
    "consistency_rate": round(float(lsi_consistency_mask.mean()), 4)
}

print(json.dumps(lsi_consistency_summary, indent=2))

reference_summary[
    "internal_consistency"
] = lsi_consistency_summary

{
  "formula_checked": "Labour Shortage Index = mean(LSI1, LSI2, LSI3)",
  "records_checked": 156,
  "consistent_records": 156,
  "inconsistent_records": 0,
  "consistency_rate": 1.0
}


In [16]:
# ------------------------------------------------------------
# 15. Reference artefact export
# ------------------------------------------------------------

reference_values_df.to_csv(
    OUTPUT_DIR / "D1_reference_values.csv",
    index=False
)

reference_values_df.to_json(
    OUTPUT_DIR / "D1_reference_values.json",
    orient="records",
    indent=2,
    force_ascii=False
)

with open(
    OUTPUT_DIR / "D1_reference_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        REFERENCE_SCHEMA,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    OUTPUT_DIR / "D1_reference_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reference_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    OUTPUT_DIR /
    "D1_internal_consistency_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        lsi_consistency_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

In [17]:
# ------------------------------------------------------------
# 16. Indicator-level assessment table
# ------------------------------------------------------------

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Low",
        "Evidence Source": "Manual inspection + workbook profiling",
        "Justification":
            "Each worksheet follows a clear top-to-bottom tabular sequence."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Medium",
        "Evidence Source": "Workbook profiling",
        "Justification":
            "Rows and columns are identifiable, but merged cells and hierarchical labels require reconstruction."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Medium",
        "Evidence Source": "Workbook profiling + manual inspection",
        "Justification":
            "Headers are explicit, but broader occupation categories are represented through merged cells and worksheet divisions."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source": "File-format inspection",
        "Justification":
            "The workbook contains natively machine-readable content; no image-quality limitation affects text recovery."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source": "File-format inspection",
        "Justification":
            "No scanning noise, degradation, or visual artefacts affect the machine-readable workbook content."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source": "File-format inspection",
        "Justification":
            "Relevant content is embedded as machine-readable spreadsheet data and OCR is not required."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source": "Header and content inspection",
        "Justification":
            "Occupation and indicator terminology is used consistently throughout the workbook, apart from one isolated source-header typo."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source": "Reference-schema comparison",
        "Justification":
            "Most source columns correspond directly to the extraction schema, but worksheet identity and hierarchical occupation labels are implicit in the native representation."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "High",
        "Evidence Source": "Automated workbook profiling",
        "Justification":
            "Approximately 80% of populated cells contain numerical or quantitative information, producing a high concentration of values requiring correct contextual association."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source": "Reference-value verification",
        "Justification":
            "All information required by the extraction task is present in the source; apparent blanks correspond to merged hierarchical labels rather than absent information."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Low",
        "Evidence Source": "Automated consistency verification",
        "Justification":
            "The schema is repeated consistently across worksheets and all 156 reported Labour Shortage Index values agree with the mean of LSI1, LSI2, and LSI3."
    },

    {
        "Dimension": "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "Medium",
        "Evidence Source": "Workbook profiling",
        "Justification":
            "The workbook contains four worksheets and merged hierarchical structures, but all sheets follow the same tabular schema."
    },
    {
        "Dimension": "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "Medium",
        "Evidence Source": "Header and representation inspection",
        "Justification":
            "Most labels and conventions are consistent, but one header typo, implicit worksheet identifiers, and merged hierarchical labels require limited standardisation."
    }
]

indicator_assessment_df = pd.DataFrame(indicator_assessment)

indicator_assessment_df

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Low,Manual inspection + workbook profiling,Each worksheet follows a clear top-to-bottom t...
1,Structural Readiness,Table Structure Integrity,Medium,Workbook profiling,"Rows and columns are identifiable, but merged ..."
2,Structural Readiness,Section/Header Hierarchy,Medium,Workbook profiling + manual inspection,"Headers are explicit, but broader occupation c..."
3,Visual/OCR Readiness,Sharpness,Low,File-format inspection,The workbook contains natively machine-readabl...
4,Visual/OCR Readiness,Noise / Degradation,Low,File-format inspection,"No scanning noise, degradation, or visual arte..."
5,Visual/OCR Readiness,OCR Dependency,Low,File-format inspection,Relevant content is embedded as machine-readab...
6,Semantic Quality,Terminology Consistency,Low,Header and content inspection,Occupation and indicator terminology is used c...
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,Most source columns correspond directly to the...
8,Semantic Quality,Numerical Density,High,Automated workbook profiling,Approximately 80% of populated cells contain n...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All information required by the extraction tas...


In [18]:
# ------------------------------------------------------------
# 17. Validation of assessment scale
# ------------------------------------------------------------

VALID_SCORES = {"Low", "Medium", "High"}

invalid_scores = set(
    indicator_assessment_df["Score"].dropna().unique()
) - VALID_SCORES

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: {invalid_scores}"
    )

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df["Indicator"]
)

missing_indicators = (
    expected_indicators - observed_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: {missing_indicators}"
    )

unexpected_indicators = (
    observed_indicators - expected_indicators
)

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: {unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print("Indicator assessment validation passed.")

Indicator assessment validation passed.


In [19]:
# ------------------------------------------------------------
# 18. Dimension-level assessment from indicator scores
# ------------------------------------------------------------

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)

dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)

def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(classify_dimension_score)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

dimension_assessment_df

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.00,2,Low
1,Representation and Normalisation Complexity,2.00,2,Medium
2,Semantic Quality,2.00,3,Medium
3,Structural Readiness,1.67,3,Medium
4,Visual/OCR Readiness,1.00,3,Low


In [20]:
# ------------------------------------------------------------
# 19. Quality-assessment artefact export
# ------------------------------------------------------------

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    OUTPUT_DIR /
    "D1_indicator_assessment.csv",
    index=False
)

dimension_assessment_df.to_csv(
    OUTPUT_DIR /
    "D1_dimension_assessment.csv",
    index=False
)

quality_evidence = {
    "document_id": DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the predefined "
        "Low, Medium, and High operational criteria defined in "
        "Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling where "
        "measurable characteristics could be derived programmatically "
        "and through documented manual inspection where qualitative "
        "assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },
        "aggregation":
            "Arithmetic mean of indicator scores within each dimension.",
        "classification_rule": {
            "Low": "mean < 1.5",
            "Medium": "1.5 <= mean < 2.5",
            "High": "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

with open(
    OUTPUT_DIR /
    "D1_quality_evidence.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        quality_evidence,
        f,
        indent=2,
        ensure_ascii=False
    )


In [21]:
# ------------------------------------------------------------
# 20. Reference metadata and integrity summary
# ------------------------------------------------------------


reference_integrity_passed = all([
    len(reference_values_df) == 156,
    not missing_columns,
    lsi_consistency_summary["inconsistent_records"] == 0
])

REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "expected_record_count": 156,
    "observed_record_count": int(len(reference_values_df)),
    "required_columns_present": bool(not missing_columns),
    "internal_consistency_passed": bool(
        lsi_consistency_summary["inconsistent_records"] == 0
    ),
    "reference_integrity_passed": bool(reference_integrity_passed)
}

if not reference_integrity_passed:
    raise ValueError(
        "D1 reference-value integrity checks failed."
    )

REFERENCE_INTEGRITY

with open(
    OUTPUT_DIR / "D1_reference_integrity.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        REFERENCE_INTEGRITY,
        f,
        indent=2,
        ensure_ascii=False
    )

In [22]:
# ------------------------------------------------------------
# 21. Final notebook summary
# ------------------------------------------------------------

summary = {
    "document_metadata": document_metadata,
    "reference_summary": reference_summary,
    "internal_consistency_summary":
        lsi_consistency_summary,
    "outputs_created": [
        "D1_sheet_characterisation.csv",
        "D1_reference_integrity.json",
        "D1_header_inspection.csv",
        "D1_extraction_schema.json",
        "D1_merged_cell_diagnostics.csv",
        "D1_merged_cell_summary.csv",
        "D1_structural_blank_summary.csv",
        "D1_reference_values.csv",
        "D1_reference_schema.json",
        "D1_reference_summary.json",
        "D1_internal_consistency_summary.json",
        "D1_hierarchical_merge_inspection.csv",
        "D1_indicator_assessment.csv",
        "D1_dimension_assessment.csv",
        "D1_reference_values.json",
        "D1_document_metadata.json",
        "D1_extraction_task.txt",
        "D1_quality_evidence.json"
]
}

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_metadata": {
    "document_id": "D1",
    "document_name": "CEDEFOP Labour Skills Shortage Dataset",
    "source_filename": "D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx",
    "source_file_sha256": "4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389",
    "file_format": "XLSX",
    "number_of_sheets": 4,
    "sheet_names": [
      "EU27",
      "IT",
      "NL",
      "PT"
    ],
    "python_version": "3.13.15",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "pandas_version": "2.2.3",
    "openpyxl_version": "3.1.5"
  },
  "reference_summary": {
    "number_of_reference_records": 156,
    "geographic_areas": [
      "EU27",
      "IT",
      "NL",
      "PT"
    ],
    "number_of_geographic_areas": 4,
    "number_of_main_occupation_groups": 4,
    "number_of_occupation_groups": 40,
    "missing_values_by_column": {
      "Geographic Area": 0,
      "Main Occupation Group": 0,
      "Occupation Group (2 digit)": 0,
      "La